In [ ]:
######  Vanilla model
import gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import pandas as pd
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_
# ======================
# 🧠 Hyperparameters
# ======================
gamma = 0.99
hidden_dim = 128
lr = 1e-3
episodes = 1000
constraint_threshold = 100 # b in constraint C(s) ≤ b
dual_lr = 5e-3              # learning rate for λ
data_van = {'cost': [], 'reward': []}
# ======================
# 🧠 Environment
# ======================
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# ======================
# 🧠 Neural Networks
# ======================
class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.policy = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
            nn.Softmax(dim=-1)
        )
    def forward(self, state):
        return self.policy(state)

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.value = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, state):
        return self.value(state)

# ======================
# 📦 Initialize
# ======================
actor = Actor()
value_critic = Critic()       # for reward
cost_critic = Critic()        # for constraint

opt_actor = optim.Adam(actor.parameters(), lr=lr)
opt_value = optim.Adam(value_critic.parameters(), lr=lr)
opt_cost = optim.Adam(cost_critic.parameters(), lr=lr)

lambda_dual = torch.tensor(10.0, requires_grad=False)  # dual variable (Lagrange multiplier)

# ======================
# 🚀 Training Loop
# ======================
start_time = time.time()
for ep in range(episodes):
    state, _ = env.reset()
    state = torch.FloatTensor(state)

    log_probs, rewards, costs = [], [], []
    values, cost_values = [], []

    done = False
    total_reward = 0.0
    total_cost = 0.0

    while not done:
        probs = actor(state)
        dist = Categorical(probs)
        action = dist.sample()

        next_state, reward, done, truncated, _ = env.step(action.item())
        done = done or truncated
        next_state = torch.FloatTensor(next_state)

        # Constraint cost: cart's distance from center
        cost = abs(state[0].item())

        # Store
        log_probs.append(dist.log_prob(action))
        rewards.append(reward)
        costs.append(cost)
        values.append(value_critic(state))
        cost_values.append(cost_critic(state))

        total_reward += reward
        total_cost += cost

        state = next_state

    # ======================
    # 🎯 Discounted Returns
    # ======================
    def discounted(x):
        ret, g = [], 0
        for r in reversed(x):
            g = r + gamma * g
            ret.insert(0, g)
        return torch.FloatTensor(ret)

    R = discounted(rewards)
    C = discounted(costs)
    V = torch.cat(values).squeeze()
    CV = torch.cat(cost_values).squeeze()
    log_probs = torch.stack(log_probs)

    # ======================
    # 🔁 Advantages
    # ======================
    A_r = R - V.detach()
    A_c = C - CV.detach()

    # ======================
    # 🎯 Policy Loss (Primal-Dual)
    # ======================
    actor_loss = -(log_probs * (A_r - lambda_dual * (A_c-b))).mean()

    # ======================
    # 🎯 Critic Losses
    # ======================
    value_loss = nn.functional.mse_loss(V, R)
    cost_loss = nn.functional.mse_loss(CV, C)

    # ======================
    # 🧠 Optimize
    # ======================
    opt_actor.zero_grad()
    actor_loss.backward()
    opt_actor.step()

    opt_value.zero_grad()
    value_loss.backward()
    opt_value.step()

    opt_cost.zero_grad()
    cost_loss.backward()
    opt_cost.step()

    # ======================
    # 🔧 Update λ (Dual Ascent)
    # ======================
    constraint_violation = (C.mean().item() - constraint_threshold)
    lambda_dual += dual_lr * constraint_violation
    lambda_dual = torch.clamp(lambda_dual, min=0.0)

    # ======================
    # 📊 Logging
    # ======================
    if (ep + 1) % 10 == 0:
        print(f"[Ep {ep+1}] Reward: {total_reward:.1f}, Cost: {total_cost:.2f}, λ: {lambda_dual.item():.3f}, Actor Loss: {actor_loss.item():.3f}")
print("Time taken", time.time() - start_time)
'''df = pd.DataFrame(data_van)
df.to_excel('tvf_and_tcf_data_vanilla.xlsx')
#Save the actors and critic models
torch.save(actor.state_dict(), 'actor_vanilla.pth')
torch.save(value_critic.state_dict(), 'reward_critic_vanilla.pth')
torch.save(cost_critic.state_dict(), 'cost_critic_vanilla.pth')'''

/usr/local/lib/python3.11/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.11/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


NameError: name 'b' is not defined

In [ ]:
!wget https://github.com/PKU-Alignment/safety-gymnasium/archive/refs/heads/main.zip
!unzip main.zip
%cd safety-gymnasium-main
!pip install -e .

--2025-07-10 18:08:11--  https://github.com/PKU-Alignment/safety-gymnasium/archive/refs/heads/main.zip
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/PKU-Alignment/safety-gymnasium/zip/refs/heads/main [following]
--2025-07-10 18:08:11--  https://codeload.github.com/PKU-Alignment/safety-gymnasium/zip/refs/heads/main
Resolving codeload.github.com (codeload.github.com)... 140.82.114.10
Connecting to codeload.github.com (codeload.github.com)|140.82.114.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘main.zip’

main.zip                [           <=>      ] 551.53M  9.22MB/s    in 28s     

2025-07-10 18:08:39 (19.9 MB/s) - ‘main.zip’ saved [578325046]

Archive:  main.zip
bfa1c945aafcd65a6b568f95d63ed9b2670046ba
   creating: safety-gymnasium-main/
  inflating: sa

In [ ]:
!pip install mujoco

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 13.4 MB/s eta 0:00:00


In [ ]:
!python setup.py

Traceback (most recent call last):
  File "/content/safety-gymnasium-main/setup.py", line 21, in <module>
    from setuptools import setup
ModuleNotFoundError: No module named 'setuptools'


In [ ]:
# # Install Python 3.8
# !sudo apt-get update -y
# !sudo apt-get install python3.8 python3.8-dev python3.8-distutils python3.8-gdbm python3.8-venv -y

# # Update symbolic links to use Python 3.8
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.8 1
!sudo update-alternatives --config python3

# Verify the Python version
!python3 --version

There are 3 choices for the alternative python3 (providing /usr/bin/python3).

  Selection    Path                 Priority   Status
------------------------------------------------------------
  0            /usr/bin/python3.11   2         auto mode
  1            /usr/bin/python3.10   1         manual mode
  2            /usr/bin/python3.11   2         manual mode
* 3            /usr/bin/python3.8    1         manual mode

Press <enter> to keep the current choice[*], or type selection number: 1
update-alternatives: using /usr/bin/python3.10 to provide /usr/bin/python3 (python3) in manual mode
Python 3.10.12


In [ ]:
max(2,3)

3

In [1]:
import gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import numpy as np
import pandas as pd
from copy import deepcopy

if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_

# === Hyperparameters ===
gamma = 0.99
hidden_dim = 256
learning_rate = 1e-3
episodes = 1000
lambda_fixed = 20.0
b = 200.0
perturb_eps = 1.0

# === Environment ===
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

# === Actor and Critic Networks ===
class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
            nn.Softmax(dim=-1)
        )
    def forward(self, state):
        return self.model(state)

class ValueCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, state):
        return self.model(state)

# === Initialize Networks and Optimizers ===
actor = Actor()
reward_critic = ValueCritic()
cost_critic = ValueCritic()

actor_optim = optim.Adam(actor.parameters(), lr=learning_rate)
reward_optim = optim.Adam(reward_critic.parameters(), lr=learning_rate)
cost_optim = optim.Adam(cost_critic.parameters(), lr=learning_rate)

# === Utilities ===
def add_uniform_noise(state, eps=0.05):
    noise = np.random.uniform(0, eps, size=state.shape)
    return state + noise

def discount(values, gamma):
    result = []
    G = 0
    for v in reversed(values):
        G = v + gamma * G
        result.insert(0, G)
    return torch.FloatTensor(result)

# === Tracking ===
dataF = {'cost': [], 'reward': []}
last_50_actor_params = []

best_reward = float('-inf')
best_actor_state_dict = None

# === Training Loop ===
for ep in range(episodes):
    state,_ = env.reset()
    state = add_uniform_noise(np.array(state), perturb_eps)
    state = torch.FloatTensor(state)

    log_probs = []
    rewards = []
    costs = []
    reward_values = []
    cost_values = []

    total_reward = 0
    total_cost = 0
    done = False

    while not done:
        probs = actor(state)
        dist = Categorical(probs)
        action = dist.sample()

        next_state, reward, done, truncated, _ = env.step(action.item())
        done = done or truncated
        next_state = add_uniform_noise(np.array(next_state), perturb_eps)
        next_state = torch.FloatTensor(next_state)

        cost = abs(state[0].item())

        log_probs.append(dist.log_prob(action))
        rewards.append(reward)
        costs.append(cost)
        reward_values.append(reward_critic(state))
        cost_values.append(cost_critic(state))

        total_reward += reward
        total_cost += cost
        state = next_state

    # Discounted returns
    reward_returns = discount(rewards, gamma)
    cost_returns = discount(costs, gamma)

    reward_values = torch.cat(reward_values).squeeze()
    cost_values = torch.cat(cost_values).squeeze()
    log_probs = torch.stack(log_probs)

    adv_r = reward_returns - reward_values.detach()
    adv_c = cost_returns - cost_values.detach()

    chosen_adv = []
    for vr, vc, ar, ac in zip(reward_returns, cost_returns, adv_r, adv_c):
        if vr.item() > lambda_fixed * (vc.item() - b):
            chosen_adv.append(ar)
        else:
            chosen_adv.append(-ac)
    chosen_adv = torch.stack(chosen_adv)

    # Losses
    actor_loss = -(log_probs * chosen_adv).mean()
    reward_loss = nn.functional.mse_loss(reward_values, reward_returns)
    cost_loss = nn.functional.mse_loss(cost_values, cost_returns)

    # Backprop
    actor_optim.zero_grad()
    actor_loss.backward()
    actor_optim.step()

    reward_optim.zero_grad()
    reward_loss.backward()
    reward_optim.step()

    cost_optim.zero_grad()
    cost_loss.backward()
    cost_optim.step()

    # Logging
    dataF['cost'].append(total_cost)
    dataF['reward'].append(total_reward)

    # Store for averaging
    if len(last_50_actor_params) >= 50:
        last_50_actor_params.pop(0)
    last_50_actor_params.append(deepcopy(actor.state_dict()))

    # === Track Best Actor ===
    if total_cost < b and total_reward > best_reward:
        best_reward = total_reward
        best_actor_state_dict = deepcopy(actor.state_dict())

    # Display
    if (ep + 1) % 50 == 0:
        print(f"Ep {ep+1} | Reward: {total_reward:.1f} | Cost: {total_cost:.2f} | Actor Loss: {actor_loss.item():.3f} | Best Reward (under cost): {best_reward:.1f}")

# === Save Averaged Actor (last 50 episodes) ===
avg_actor_state_dict = deepcopy(last_50_actor_params[0])
for key in avg_actor_state_dict:
    for i in range(1, len(last_50_actor_params)):
        avg_actor_state_dict[key] += last_50_actor_params[i][key]
    avg_actor_state_dict[key] /= len(last_50_actor_params)

avg_actor = Actor()
avg_actor.load_state_dict(avg_actor_state_dict)

# === Save All Models ===
env.close()
df = pd.DataFrame(dataF)
df.to_excel('tvf_and_tcf_data_with_uncertainity.xlsx')

torch.save(actor.state_dict(), 'actor.pth')                       # Final actor
torch.save(avg_actor.state_dict(), 'actor_avg_last50.pth')       # Averaged actor
torch.save(reward_critic.state_dict(), 'reward_critic.pth')
torch.save(cost_critic.state_dict(), 'cost_critic.pth')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Ep 50 | Reward: 20.0 | Cost: 8.50 | Actor Loss: 4.758 | Best Reward (under cost): 81.0
Ep 100 | Reward: 41.0 | Cost: 20.06 | Actor Loss: 6.798 | Best Reward (under cost): 101.0
Ep 150 | Reward: 56.0 | Cost: 31.64 | Actor Loss: 4.160 | Best Reward (under cost): 149.0
Ep 200 | Reward: 44.0 | Cost: 18.90 | Actor Loss: 0.670 | Best Reward (under cost): 149.0
Ep 250 | Reward: 79.0 | Cost: 32.26 | Actor Loss: 5.716 | Best Reward (under cost): 159.0
Ep 300 | Reward: 42.0 | Cost: 23.16 | Actor Loss: -10.171 | Best Reward (under cost): 193.0
Ep 350 | Reward: 53.0 | Cost: 20.49 | Actor Loss: -4.818 | Best Reward (under cost): 269.0
Ep 400 | Reward: 59.0 | Cost: 30.59 | Actor Loss: -4.603 | Best Reward (under cost): 269.0
Ep 450 | Reward: 276.0 | Cost: 289.93 | Actor Loss: 8.374 | Best Reward (under cost): 269.0
Ep 500 | Reward: 150.0 | Cost: 156.20 | Actor Loss: 0.804 | Best Reward (under cost): 269.0
Ep 550 | Reward: 134.0 | Cost: 49.85 | Actor Loss: 3.187 | Best Reward (under cost): 307.0
Ep 6